# Fase 2: integración

Une en **una sola tabla** todo lo que sale de la limpieza: los sensores (`tables`),
los PIR (`pir`) y la etiqueta (`labels`). Como todo comparte la misma rejilla de
tiempo, cada fila será un instante (cada 10 s) con todas sus medidas y su nivel.

Este notebook no repite la limpieza: la toma ya hecha con la función `clean` de
`src/occupancy/cleaning.py`.

Diapositivas: clase 7, 24 (Data Integration) y 27–33 (redundancia y correlación).

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
import yaml

# Raíz del repositorio y acceso al código de src/
ROOT = Path.cwd()
while not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from occupancy.cleaning import clean

with open(ROOT / "config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)

with open(ROOT / "data" / "simulated" / "readings.json", encoding="utf-8") as f:
    data = json.load(f)["data"]

# Fase 1 completa en una línea
tables, pir, labels = clean(data, config["cleaning"])

print("Sensores:", list(tables))
print("PIR:", list(pir.columns))
print("Etiqueta:", list(labels.columns))

Sensores: ['pressure', 'humidity', 'temperature', 'co2Equivalent', 'voc', 'iaq', 'mic_vol', 'wifiPackets', 'bleAdverts']
PIR: ['timestamp', 'pir-1', 'pir-2', 'pir-3']
Etiqueta: ['timestamp', 'busynessLabel']


## Tu tarea: unir todo en una tabla

Guarda el resultado en una tabla llamada **`df`**:

- **Índice:** `timestamp`.
- **Columnas:** una por sensor (con el nombre del sensor), `pir-1`, `pir-2`, `pir-3`
  y `busynessLabel`.

Pasos:

1. Convierte cada tabla de `tables` en una serie con `timestamp` como índice
   (`set_index`) y quédate con su columna `value`, guardándola con el nombre del sensor.
2. Haz lo mismo con `pir` y `labels`: `set_index("timestamp")` y ya tienen sus columnas.
3. Únelas todas por el índice con `pd.concat([...], axis=1)`.

`axis=1` significa "pegar al lado" (como columnas), alineando las filas por su índice.

In [ ]:
# Tu código aquí
pf=None
parts = []
for sensor, table in tables.items():
    column = table.set_index("timestamp")["value"].rename(sensor)
    parts.append(column)




In [ ]:
# Comprobación de la integración (no hace falta modificarla)
expected_columns = set(tables) | {"pir-1", "pir-2", "pir-3", "busynessLabel"}
checks = {
    "1080 filas": len(df) == 1080,
    "índice de fechas": isinstance(df.index, pd.DatetimeIndex),
    "columnas correctas": set(df.columns) == expected_columns,
    "sin horas repetidas": not df.index.duplicated().any(),
    "21 filas sin etiqueta": df["busynessLabel"].isna().sum() == 21,
}
for name, ok in checks.items():
    print(f"{name:22} {'OK' if ok else 'REVISAR'}")

## Mirar la tabla

Antes de seguir, conviene ver qué ha quedado. Prueba con:

- `df.head()`: las primeras filas.
- `df.describe()`: mínimo, máximo y media de cada columna.
- `df.isna().sum()`: cuántos vacíos tiene cada columna.
- `df["busynessLabel"].value_counts()`: cuántas filas hay de cada nivel.

In [ ]:
# Tu código aquí

## Redundancia: correlación entre columnas

La diapositiva 27 habla de atributos redundantes, y la 31 de cómo detectarlos con la
**correlación**. `df.corr()` calcula la correlación de cada pareja de columnas
(de −1 a 1). Dos sensores con correlación muy cercana a 1 dicen casi lo mismo.

**Aviso:** con los datos simulados todo saldrá muy correlacionado, porque todos los
sensores se generaron a partir del número de personas. Aquí solo practicas cómo se
hace; las conclusiones se sacan con datos reales (fase de reducción).

In [ ]:
# Tu código aquí